In [ ]:
import numpy as np
import json

# Load embeddings
E_softmax = np.load("embeddings/sg_softmax_embeddings.npy")
E_neg = np.load("embeddings/sg_neg_embeddings.npy")
E_glove = np.load("embeddings/glove_embeddings.npy")

print("Loaded shapes:", E_softmax.shape, E_neg.shape, E_glove.shape)

def normalize_rows(M, eps=1e-9):
    norms = np.linalg.norm(M, axis=1, keepdims=True)
    return M / (norms + eps)

# normalized versions for fast cosine
E_softmax_n = normalize_rows(E_softmax)
E_neg_n = normalize_rows(E_neg)
E_glove_n = normalize_rows(E_glove)

# Load vocabulary
with open("data/word2id.json", "r") as f:
    word2id = json.load(f)

print("Vocabulary size:", len(word2id))

In [ ]:
def analogy_accuracy(questions, E_n, word2id, topk=1):
    """
    questions: list of (a,b,c,d) where a:b :: c:d
    Return accuracy@topk over questions that are all in vocab.
    """
    correct = 0
    used = 0
    for a,b,c,d in questions:
        if a not in word2id or b not in word2id or c not in word2id or d not in word2id:
            continue
        ia, ib, ic, id_ = word2id[a], word2id[b], word2id[c], word2id[d]
        # v = b - a + c
        v = E_n[ib] - E_n[ia] + E_n[ic]
        v = v / (np.linalg.norm(v) + 1e-9)

        sims = E_n @ v  # cosine because normalized
        # exclude input words
        sims[[ia, ib, ic]] = -1e9
        # topk predictions
        pred_ids = np.argpartition(-sims, topk)[:topk]
        if id_ in pred_ids:
            correct += 1
        used += 1
    acc = correct / used if used > 0 else 0.0
    return acc, used

print("Analogy evaluation function defined")

In [ ]:
def load_analogies(filepath, category):
    """Load specific category from questions-words.txt"""
    questions = []
    in_category = False
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip().lower()
            
            if line.startswith(':'):
                if category.lower() in line:
                    in_category = True
                else:
                    in_category = False
                continue
            
            if in_category and line:
                parts = line.split()
                if len(parts) == 4:
                    questions.append(tuple(parts))
    
    return questions

# Load semantic (capital-common-countries)
semantic_questions = load_analogies('data/questions-words.txt', 'capital-common-countries')
print(f"Semantic questions: {len(semantic_questions)}")

# Load syntactic (past-tense)
syntactic_questions = load_analogies('data/questions-words.txt', 'gram7-past-tense')
print(f"Syntactic questions: {len(syntactic_questions)}")

In [ ]:
print("Evaluating Skip-gram (Softmax)...")
sem_acc_sg, sem_used_sg = analogy_accuracy(semantic_questions, E_softmax_n, word2id)
syn_acc_sg, syn_used_sg = analogy_accuracy(syntactic_questions, E_softmax_n, word2id)

print(f"  Semantic: {sem_acc_sg:.4f} (used {sem_used_sg}/{len(semantic_questions)})")
print(f"  Syntactic: {syn_acc_sg:.4f} (used {syn_used_sg}/{len(syntactic_questions)})")

In [ ]:
print("Evaluating Skip-gram (NEG)...")
sem_acc_neg, sem_used_neg = analogy_accuracy(semantic_questions, E_neg_n, word2id)
syn_acc_neg, syn_used_neg = analogy_accuracy(syntactic_questions, E_neg_n, word2id)

print(f"  Semantic: {sem_acc_neg:.4f} (used {sem_used_neg}/{len(semantic_questions)})")
print(f"  Syntactic: {syn_acc_neg:.4f} (used {syn_used_neg}/{len(syntactic_questions)})")

In [ ]:
print("Evaluating GloVe...")
sem_acc_glove, sem_used_glove = analogy_accuracy(semantic_questions, E_glove_n, word2id)
syn_acc_glove, syn_used_glove = analogy_accuracy(syntactic_questions, E_glove_n, word2id)

print(f"  Semantic: {sem_acc_glove:.4f} (used {sem_used_glove}/{len(semantic_questions)})")
print(f"  Syntactic: {syn_acc_glove:.4f} (used {syn_used_glove}/{len(syntactic_questions)})")

In [ ]:
import gensim.downloader as api

print("Loading pre-trained GloVe from Gensim...")
try:
    glove_pretrained = api.load("glove-wiki-gigaword-50")
    print("Loaded successfully")
    
    # Convert to our format
    E_pretrained = []
    pretrained_word2id = {}
    
    for i, word in enumerate(word2id.keys()):
        if word in glove_pretrained:
            E_pretrained.append(glove_pretrained[word])
            pretrained_word2id[word] = i
        else:
            E_pretrained.append(np.zeros(50))
            pretrained_word2id[word] = i
    
    E_pretrained = np.array(E_pretrained)
    E_pretrained_n = normalize_rows(E_pretrained)
    
    print("Evaluating pre-trained GloVe...")
    sem_acc_pre, sem_used_pre = analogy_accuracy(semantic_questions, E_pretrained_n, pretrained_word2id)
    syn_acc_pre, syn_used_pre = analogy_accuracy(syntactic_questions, E_pretrained_n, pretrained_word2id)
    
    print(f"  Semantic: {sem_acc_pre:.4f} (used {sem_used_pre}/{len(semantic_questions)})")
    print(f"  Syntactic: {syn_acc_pre:.4f} (used {syn_used_pre}/{len(syntactic_questions)})")
    
except Exception as e:
    print(f"Could not load: {e}")
    sem_acc_pre = syn_acc_pre = 0.0

In [ ]:
from scipy.stats import spearmanr

# Create sample WordSim353 if not exists
import os
if not os.path.exists('data/wordsim353.tsv'):
    with open('data/wordsim353.tsv', 'w') as f:
        f.write("""love\tsex\t6.77
tiger\tcat\t7.35
book\tpaper\t7.46
computer\tkeyboard\t7.62
plane\tcar\t5.77
doctor\tnurse\t7.00
smart\tintelligent\t9.20
king\tqueen\t8.58
money\tcash\t9.15
bank\tmoney\t8.12""")

def load_wordsim353(filepath):
    pairs = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            if len(parts) >= 3:
                try:
                    w1, w2, score = parts[0].lower(), parts[1].lower(), float(parts[2])
                    pairs.append((w1, w2, score))
                except:
                    continue
    return pairs

def wordsim_spearman(wordsim_pairs, E_n, word2id):
    human_scores = []
    model_scores = []

    for w1, w2, score in wordsim_pairs:
        if w1 not in word2id or w2 not in word2id:
            continue
        i, j = word2id[w1], word2id[w2]
        sim = float(np.dot(E_n[i], E_n[j]))
        human_scores.append(score)
        model_scores.append(sim)

    if len(human_scores) < 2:
        return 0.0, 0
    
    corr, _ = spearmanr(human_scores, model_scores)
    return corr, len(human_scores)

wordsim_pairs = load_wordsim353('data/wordsim353.tsv')
print(f"Loaded {len(wordsim_pairs)} word pairs")

# Compute for all 3 models
ws_softmax, n1 = wordsim_spearman(wordsim_pairs, E_softmax_n, word2id)
ws_neg, n2 = wordsim_spearman(wordsim_pairs, E_neg_n, word2id)
ws_glove, n3 = wordsim_spearman(wordsim_pairs, E_glove_n, word2id)

print("\nWordSim353 Spearman:")
print(f"Skip-gram Softmax: {ws_softmax:.4f} (pairs used={n1})")
print(f"Skip-gram NEG    : {ws_neg:.4f} (pairs used={n2})")
print(f"GloVe            : {ws_glove:.4f} (pairs used={n3})")

In [ ]:
# Load training results
with open("results/training_results.json", "r") as f:
    training_results = json.load(f)

# Combined results
results = {
    'skipgram': {
        'window_size': training_results['skipgram']['window_size'],
        'training_time': training_results['skipgram']['training_time'],
        'training_loss': training_results['skipgram']['final_loss'],
        'semantic_accuracy': sem_acc_sg,
        'syntactic_accuracy': syn_acc_sg,
        'wordsim_correlation': ws_softmax
    },
    'skipgram_neg': {
        'window_size': training_results['skipgram_neg']['window_size'],
        'training_time': training_results['skipgram_neg']['training_time'],
        'training_loss': training_results['skipgram_neg']['final_loss'],
        'semantic_accuracy': sem_acc_neg,
        'syntactic_accuracy': syn_acc_neg,
        'wordsim_correlation': ws_neg
    },
    'glove': {
        'window_size': training_results['glove']['window_size'],
        'training_time': training_results['glove']['training_time'],
        'training_loss': training_results['glove']['final_loss'],
        'semantic_accuracy': sem_acc_glove,
        'syntactic_accuracy': syn_acc_glove,
        'wordsim_correlation': ws_glove
    },
    'glove_pretrained': {
        'window_size': '-',
        'training_time': '-',
        'training_loss': '-',
        'semantic_accuracy': sem_acc_pre,
        'syntactic_accuracy': syn_acc_pre,
        'wordsim_correlation': '-'
    }
}

with open("results/evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Results saved to results/evaluation_results.json")
print("\n" + "=" * 100)
print("RESULTS SUMMARY TABLE")
print("=" * 100)
print(f"{'Model':<20} {'Window':<8} {'Train Loss':<12} {'Train Time':<12} {'Syntactic':<12} {'Semantic':<12}")
print("=" * 100)

for model_name, model_results in results.items():
    w = str(model_results['window_size'])
    loss = f"{model_results['training_loss']:.4f}" if isinstance(model_results['training_loss'], (int, float)) else model_results['training_loss']
    time_val = f"{model_results['training_time']:.2f}s" if isinstance(model_results['training_time'], (int, float)) else model_results['training_time']
    syn = f"{model_results['syntactic_accuracy']:.4f}"
    sem = f"{model_results['semantic_accuracy']:.4f}"
    
    print(f"{model_name:<20} {w:<8} {loss:<12} {time_val:<12} {syn:<12} {sem:<12}")

print("=" * 100)